# 01 — Ingestão de Dados

**Objetivo:** Download automático dos CSVs da ANP (série histórica de preços de combustíveis) e coleta de dados de enriquecimento (dólar e Brent).

**Fonte primária:** [ANP — Série Histórica de Preços](https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/serie-historica-de-precos-de-combustiveis)

**Pipeline:**
1. Descobrir links de CSV na página da ANP
2. Download de todos os semestres (2004–2025)
3. Coleta de cotação do dólar (API do Banco Central)
4. Coleta do preço do petróleo Brent (EIA/FRED)
5. Upload para AWS S3 (raw/)

**Regras do projeto:**
- Somente CSVs semestrais (2004?2025).
- CSVs mensais e 2026 s?o ignorados.
- Upload para AWS S3 ? opcional e n?o ? requisito.


In [1]:
import sys
sys.path.insert(0, '..')

from src.ingestao import descobrir_links_csv, download_todos_csvs, upload_todos_para_s3
from src.scraping import obter_cotacao_dolar, obter_preco_brent, salvar_dados_externos
from src.utils import DATA_RAW, DATA_EXTERNAL

## 1.1 — Descobrir links de download na ANP

In [2]:
links = descobrir_links_csv()
print(f"Encontrados {len(links)} arquivos CSV")
for link in links[:5]:
    print(f"  {link['semestre']}: {link['nome']}")
print(f"  ...")

13:33:34 - ingestao - INFO - Buscando links de CSV na página da ANP...
13:33:35 - ingestao - INFO - Encontrados 200 arquivos CSV


Encontrados 200 arquivos CSV
  2021: 2º semestre de 2021
  2021: 1º semestre de 2021
  2020: 2º semestre de 2020
  2020: 1º semestre de 2020
  2019: 2º semestre de 2019
  ...


## 1.2 ? Download dos CSVs
Apenas semestrais (2004?2025); mensais e 2026 s?o ignorados.


In [3]:
arquivos = download_todos_csvs(DATA_RAW, ano_min=2004, ano_max=2025, apenas_semestrais=True)
print(f"\n{len(arquivos)} arquivos baixados em {DATA_RAW}")
for arq in arquivos[:5]:
    print(f"  {arq.name} — {arq.stat().st_size / 1e6:.1f} MB")

13:33:35 - ingestao - INFO - Buscando links de CSV na página da ANP...
13:33:36 - ingestao - INFO - Encontrados 200 arquivos CSV
13:33:36 - ingestao - INFO - Links encontrados: 200 | considerados: 44 | ignorados: 156
13:33:36 - ingestao - INFO - Arquivo já existe, pulando: 2º_semestre_de_2021.csv
13:33:36 - ingestao - INFO - Arquivo já existe, pulando: 1º_semestre_de_2021.csv
13:33:36 - ingestao - INFO - Arquivo já existe, pulando: 2º_semestre_de_2020.csv
13:33:36 - ingestao - INFO - Arquivo já existe, pulando: 1º_semestre_de_2020.csv
13:33:36 - ingestao - INFO - Arquivo já existe, pulando: 2º_semestre_de_2019.csv
13:33:36 - ingestao - INFO - Arquivo já existe, pulando: 1º_semestre_de_2019.csv
13:33:36 - ingestao - INFO - Arquivo já existe, pulando: 2º_semestre_de_2018.csv
13:33:36 - ingestao - INFO - Arquivo já existe, pulando: 1º_semestre_de_2018.csv
13:33:36 - ingestao - INFO - Arquivo já existe, pulando: 2º_semestre_de_2017.csv
13:33:36 - ingestao - INFO - Arquivo já existe, puland


44 arquivos baixados em c:\Users\leona\Desktop\TI\Combustivel Brasil Analytics\notebooks\..\data\raw
  2º_semestre_de_2021.csv — 39.7 MB
  1º_semestre_de_2021.csv — 57.9 MB
  2º_semestre_de_2020.csv — 2.4 MB
  1º_semestre_de_2020.csv — 86.7 MB
  2º_semestre_de_2019.csv — 89.0 MB


## 1.3 — Explorar metadados de um CSV

In [4]:
import pandas as pd

csvs = sorted(DATA_RAW.glob('*.csv'))
if csvs:
    amostra = pd.read_csv(csvs[0], sep=';', encoding='latin-1', nrows=5)
    print(f"Arquivo: {csvs[0].name}")
    print(f"Colunas: {list(amostra.columns)}")
    print(f"\nAmostra:")
    display(amostra)
else:
    print("Nenhum CSV encontrado. Execute o download primeiro.")

Arquivo: 1º_semestre_de_2004.csv
Colunas: ['ï»¿Regiao - Sigla', 'Estado - Sigla', 'Municipio', 'Revenda', 'CNPJ da Revenda', 'Nome da Rua', 'Numero Rua', 'Complemento', 'Bairro', 'Cep', 'Produto', 'Data da Coleta', 'Valor de Venda', 'Valor de Compra', 'Unidade de Medida', 'Bandeira']

Amostra:


,ï»¿Regiao - Sigla,Estado - Sigla,Municipio,Revenda,CNPJ da Revenda,Nome da Rua,Numero Rua,Complemento,Bairro,Cep,Produto,Data da Coleta,Valor de Venda,Valor de Compra,Unidade de Medida,Bandeira
0,SE,SP,GUARULHOS,AUTO POSTO SAKAMOTO LTDA,49.051.667/0001-02,RODOVIA PRESIDENTE DUTRA,S/N,"KM 210,5-SENT SP/RJ",BONSUCESSO,07178-580,GASOLINA,11/05/2004,"1,967","1,6623",R$ / litro,PETROBRAS DISTRIBUIDORA S.A.
1,SE,SP,GUARULHOS,AUTO POSTO SAKAMOTO LTDA,49.051.667/0001-02,RODOVIA PRESIDENTE DUTRA,S/N,"KM 210,5-SENT SP/RJ",BONSUCESSO,07178-580,ETANOL,11/05/2004,"0,899","0,6282",R$ / litro,PETROBRAS DISTRIBUIDORA S.A.
2,SE,SP,GUARULHOS,AUTO POSTO SAKAMOTO LTDA,49.051.667/0001-02,RODOVIA PRESIDENTE DUTRA,S/N,"KM 210,5-SENT SP/RJ",BONSUCESSO,07178-580,DIESEL,11/05/2004,"1,299","1,1704",R$ / litro,PETROBRAS DISTRIBUIDORA S.A.
3,SE,SP,SOROCABA,COMPETRO COMERCIO E DISTRIBUICAO DE DERIVADOS ...,00.003.188/0001-21,RUA HUMBERTO DE CAMPOS,306,NaN,JARDIM ZULMIRA,18061-000,GASOLINA,10/05/2004,"1,85","1,67",R$ / litro,BRANCA
4,SE,SP,SOROCABA,COMPETRO COMERCIO E DISTRIBUICAO DE DERIVADOS ...,00.003.188/0001-21,RUA HUMBERTO DE CAMPOS,306,NaN,JARDIM ZULMIRA,18061-000,ETANOL,10/05/2004,"0,78","0,48",R$ / litro,BRANCA


## 1.4 — Coleta de dados externos

In [5]:
df_dolar = obter_cotacao_dolar()
print(f"Cotação do dólar: {len(df_dolar)} dias")
print(f"Período: {df_dolar['data'].min()} a {df_dolar['data'].max()}")
df_dolar.tail()

13:33:36 - scraping - INFO - Buscando cotação do dólar de 01-01-2004 a 04-14-2026...
13:33:36 - scraping - INFO -   2004: 252 registros
13:33:37 - scraping - INFO -   2005: 251 registros
13:33:37 - scraping - INFO -   2006: 249 registros
13:33:37 - scraping - INFO -   2007: 250 registros
13:33:38 - scraping - INFO -   2008: 254 registros
13:33:38 - scraping - INFO -   2009: 250 registros
13:33:38 - scraping - INFO -   2010: 251 registros
13:33:38 - scraping - INFO -   2011: 251 registros
13:33:39 - scraping - INFO -   2012: 251 registros
13:33:39 - scraping - INFO -   2013: 253 registros
13:33:39 - scraping - INFO -   2014: 253 registros
13:33:40 - scraping - INFO -   2015: 250 registros
13:33:40 - scraping - INFO -   2016: 251 registros
13:33:40 - scraping - INFO -   2017: 249 registros
13:33:40 - scraping - INFO -   2018: 250 registros
13:33:41 - scraping - INFO -   2019: 253 registros
13:33:41 - scraping - INFO -   2020: 251 registros
13:33:41 - scraping - INFO -   2021: 251 registr

Cotação do dólar: 5595 dias
Período: 2004-01-02 00:00:00 a 2026-04-14 00:00:00


,data,cotacao_compra,cotacao_venda
5590,2026-04-08,5.0893,5.0899
5591,2026-04-09,5.0815,5.0821
5592,2026-04-10,5.0223,5.0229
5593,2026-04-13,5.0238,5.0244
5594,2026-04-14,4.9800,4.9806


In [6]:
df_brent = obter_preco_brent()
print(f"Preço Brent: {len(df_brent)} registros")
print(f"Período: {df_brent['data'].min()} a {df_brent['data'].max()}")
df_brent.tail()

13:33:42 - scraping - INFO - Buscando preço histórico do Brent (EIA)...
13:33:45 - scraping - INFO - Preço Brent: 9864 registros obtidos


Preço Brent: 9864 registros
Período: 1987-05-20 00:00:00 a 2026-04-02 00:00:00


,data,preco_brent_usd
9859,2026-03-27,121.47
9860,2026-03-30,121.88
9861,2026-03-31,126.69
9862,2026-04-01,119.56
9863,2026-04-02,127.61


In [7]:
paths = salvar_dados_externos(df_dolar, df_brent)
for nome, path in paths.items():
    print(f"{nome}: {path}")

13:33:45 - scraping - INFO - Dólar salvo: c:\Users\leona\Desktop\TI\Combustivel Brasil Analytics\notebooks\..\data\external\cotacao_dolar.parquet
13:33:45 - scraping - INFO - Brent salvo: c:\Users\leona\Desktop\TI\Combustivel Brasil Analytics\notebooks\..\data\external\preco_brent.parquet


dolar: c:\Users\leona\Desktop\TI\Combustivel Brasil Analytics\notebooks\..\data\external\cotacao_dolar.parquet
brent: c:\Users\leona\Desktop\TI\Combustivel Brasil Analytics\notebooks\..\data\external\preco_brent.parquet


## 1.5 ? Upload para AWS S3 (opcional)

Este passo ? opcional e n?o ? requisito para rodar o projeto.

?? Requer configura??o do arquivo `.env` com credenciais AWS.


In [8]:
# Descomente para executar o upload
# uris = upload_todos_para_s3(DATA_RAW)
# print(f"{len(uris)} arquivos enviados para S3")

---
**Próximo passo:** [02_etl_limpeza.ipynb](02_etl_limpeza.ipynb) — Limpeza e transformação dos dados